# 05-08 Custom Generator: 스트리밍을 유지하며 데이터 변환하기

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:

- 파이썬 제너레이터(Generator) 함수를 LCEL 파이프라인에 연결하여 스트리밍 방식으로 데이터를 변환하는 방법을 구현할 수 있어요
- 동기(Sync) 제너레이터와 비동기(Async) 제너레이터의 시그니처(Signature) 차이를 설명할 수 있어요
- 스트리밍 도중 쉼표로 구분된 문자열을 실시간으로 리스트 항목으로 변환하는 출력 파서를 만들 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- 파이썬 제너레이터 함수와 `yield` 키워드
- `Iterator`와 `AsyncIterator` 타입 힌트
- LCEL `stream()` 메서드의 동작 방식

---

제너레이터(Generator) 함수를 LCEL 파이프라인에서 사용할 수 있어요.

**제너레이터 시그니처(Signature):**
- 동기: `Iterator[Input] -> Iterator[Output]`
- 비동기: `AsyncIterator[Input] -> AsyncIterator[Output]`

**주요 활용:**
- 사용자 정의 출력 파서(Output Parser) 구현
- 스트리밍 기능을 유지하면서 이전 단계의 출력 수정
- 실시간 데이터 처리 및 변환

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

In [2]:
from typing import Iterator, List

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 프롬프트 템플릿 생성
prompt = ChatPromptTemplate.from_template(
    "{company}와 유사한 5개의 회사를 쉼표로 구분된 목록으로 작성하세요."
)

# 모델 초기화
model = ChatOpenAI(temperature=0.0, model="gpt-4o-mini")

# 문자열 체인 생성
str_chain = prompt | model | StrOutputParser()

print("=" * 60)
print("✅ 기본 체인 생성 완료")
print("=" * 60)


✅ 기본 체인 생성 완료


## 1. 기본 스트리밍

먼저 기본 스트리밍이 어떻게 동작하는지 확인해볼게요.

`stream()` 메서드는 LLM의 출력을 청크(Chunk) 단위로 실시간으로 반환해요. 사용자는 전체 응답이 완성되기 전에도 출력이 생성되는 것을 볼 수 있어요.

> **실무 팁:** 스트리밍은 응답 시간이 긴 LLM 호출에서 사용자 경험(UX)을 크게 개선해요. 전체 응답을 기다리지 않고 부분적으로 결과를 보여줄 수 있어요.

In [3]:
# 스트리밍으로 실행
print("=" * 60)
print("📥 스트리밍 출력")
print("=" * 60)
for chunk in str_chain.stream({"company": "구글"}):
    print(chunk, end="", flush=True)
print("\n")


📥 스트리밍 출력
구글과 유사한 5개의 회사는 다음과 같습니다: 마이크로소프트, 애플, 아마존, 페이스북(메타), 바이두.



In [4]:
# invoke()로 실행 결과 확인
result = str_chain.invoke({"company": "구글"})
print("=" * 60)
print("📥 invoke() 결과")
print("=" * 60)
print(result)


📥 invoke() 결과
구글과 유사한 5개의 회사는 다음과 같습니다: 마이크로소프트, 애플, 아마존, 페이스북(메타), 바이두.


## 2. 사용자 정의 제너레이터 구현

쉼표로 구분된 문자열을 리스트로 변환하는 제너레이터를 구현해요.

스트리밍 기능을 유지하면서 각 항목을 개별적으로 반환해요.

> **주의:** 제너레이터 함수를 LCEL 파이프라인에서 사용할 때는 `|` 연산자로 연결하면 돼요. LangChain이 자동으로 `Iterator` 타입 힌트를 감지하여 스트리밍 체인으로 처리해요.

In [5]:
# ============================================================
# TODO: 쉼표로 구분된 문자열 스트림을 실시간으로 리스트 항목으로 분할하는 제너레이터를 작성하세요
# 힌트: Iterator[str] → Iterator[List[str]], 버퍼에 쉼표가 나타나면 즉시 yield
# 예상 결과: 스트리밍 중에 ['마이크로소프트'], ['애플'], ... 순서로 출력
# ============================================================

# ---------------------------------------------------
# 사용자 정의 제너레이터: 스트리밍을 유지하며 출력 형식 변환
# ---------------------------------------------------

# 제너레이터 시그니처: Iterator[Input] -> Iterator[Output]
# - 입력: LLM 출력 스트림 (문자열 청크)
# - 출력: 각 항목을 포함한 리스트
def split_into_list(input: Iterator[str]) -> Iterator[List[str]]:
    """
    쉼표로 구분된 문자열 스트림을 리스트 항목 스트림으로 변환
    
    Args:
        input: 문자열 이터레이터 (LLM 출력 스트림)
    
    Yields:
        각 항목을 포함한 리스트
    """
    buffer = ""  # 쉼표를 만날 때까지 문자를 저장할 버퍼
    
    for chunk in input:
        buffer += chunk  # 현재 청크를 버퍼에 추가
        
        # 버퍼에 쉼표가 있는 동안 반복 — 쉼표를 기준으로 항목 분리
        while "," in buffer:
            comma_index = buffer.index(",")
            # 쉼표 이전의 내용을 하나의 항목으로 즉시 반환 (스트리밍 유지)
            yield [buffer[:comma_index].strip()]
            # 쉼표 이후의 내용은 다음 반복을 위해 버퍼에 유지
            buffer = buffer[comma_index + 1:]
    
    # 마지막 남은 버퍼 내용 반환 (스트림 끝에 쉼표가 없는 마지막 항목)
    if buffer.strip():
        yield [buffer.strip()]


print("=" * 60)
print(":white_check_mark: 제너레이터 함수 정의 완료")
print("=" * 60)

:white_check_mark: 제너레이터 함수 정의 완료


In [ ]:
# 제너레이터를 체인에 연결


In [ ]:
# 제너레이터 체인 스트리밍 테스트


In [ ]:
# invoke()로 실행 결과 확인


## 3. 비동기 제너레이터

비동기 제너레이터(Async Generator)를 사용하면 비동기 스트리밍도 가능해요.

`async for`를 사용하여 비동기 이터레이터를 처리하고, `astream()`으로 비동기 스트리밍을 실행해요.

In [ ]:
from typing import AsyncIterator

# ============================================================
# TODO: 동기 제너레이터를 비동기 버전으로 변환하세요
# 힌트: async def fn(input: AsyncIterator[str]) -> AsyncIterator[List[str]]: async for chunk in input: ...
# 예상 결과: astream()으로 비동기 스트리밍 출력
# ============================================================

# ---------------------------------------------------
# 비동기 제너레이터: AsyncIterator[Input] -> AsyncIterator[Output]
# ---------------------------------------------------


In [ ]:
# 비동기 스트리밍 테스트


In [ ]:
# 비동기 invoke() 테스트


## 마무리 요약

### 제너레이터 타입 비교

| 종류 | 시그니처 | 실행 메서드 |
|------|---------|------------|
| 동기 제너레이터 | `Iterator[Input] -> Iterator[Output]` | `stream()` |
| 비동기 제너레이터 | `AsyncIterator[Input] -> AsyncIterator[Output]` | `astream()` |

### 핵심 요점

- 제너레이터 함수는 `|` 연산자로 LCEL 파이프라인에 바로 연결할 수 있어요
- 스트리밍 기능을 유지하면서 중간 단계에서 출력 형식을 변환할 수 있어요
- 버퍼(Buffer) 패턴으로 구분자(쉼표 등)를 감지하고 항목을 분리해 실시간으로 반환해요
- 비동기 제너레이터는 `async def`와 `async for`로 작성하고 `astream()`으로 실행해요

### 다음 노트북 예고

다음 노트북(`09-Binding.ipynb`)에서는 `bind()` 메서드로 모델에 stop 토큰, OpenAI 함수(Function), 도구(Tool) 등을 런타임 전에 미리 바인딩(Binding)하는 방법을 배워요.